# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 3, 'num_retries': 0}


In [4]:
import json
from pathlib import Path

commit0 = Path(table_path) / "_delta_log" / "00000000000000000000.json"
print(f"Commit JSON: {commit0}")
# Delta commits are JSON Lines: each line contains one transaction action.
for line in commit0.read_text(encoding="utf-8").splitlines():
    print(json.dumps(json.loads(line), indent=2, ensure_ascii=False))

Commit JSON: C:\VinAI\K4-Track02-Day18-TrinhHoangTung-2A202602937-Lakehouse-Lab\_lakehouse\scratch\users_delta\_delta_log\00000000000000000000.json
{
  "commitInfo": {
    "timestamp": 1791132616003,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "Overwrite"
    },
    "engineInfo": "delta-rs:py-1.6.6",
    "operationMetrics": {
      "num_added_files": 1,
      "num_removed_files": 0,
      "num_partitions": 0,
      "num_added_rows": 3,
      "execution_time_ms": 3,
      "num_retries": 0
    },
    "clientVersion": "delta-rs.py-1.6.6"
  }
}
{
  "protocol": {
    "minReaderVersion": 1,
    "minWriterVersion": 2
  }
}
{
  "metaData": {
    "id": "8bf82f9e-ee46-4e0a-8c0b-19e33b4ac784",
    "name": null,
    "description": null,
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"name\",\"type\":\"string\",\"nul

## 3. Schema enforcement — try to write a wrong schema

In [5]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
before_bad = DeltaTable(table_path)
version_before_bad = before_bad.version()
data_before_bad = before_bad.to_pyarrow_table()
commits_before_bad = set(Path(table_path).glob("_delta_log/*.json"))
enforcement_blocked = False
bad_write_error = None
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    # delta-rs 1.6.6 exposes this cast failure as a generic Exception.
    # Only the actual cast error for our invalid value counts as enforcement.
    bad_write_error = f"{type(e).__name__}: {e}"
    enforcement_blocked = "thirty" in str(e) and "cast" in str(e).lower()
    print(f"BLOCKED by schema enforcement (expected): {bad_write_error}")

after_bad = DeltaTable(table_path)
bad_write_unchanged = (
    after_bad.version() == version_before_bad
    and after_bad.to_pyarrow_table().equals(data_before_bad)
    and set(Path(table_path).glob("_delta_log/*.json")) == commits_before_bad
)
print(f"Version before/after bad write: {version_before_bad}/{after_bad.version()}")
print(f"Rows before/after bad write: {data_before_bad.num_rows}/{after_bad.to_pyarrow_table().num_rows}")
print(f"Data and commit list unchanged: {bad_write_unchanged}")
assert enforcement_blocked and bad_write_unchanged, "Bad write must fail with a cast error and leave the table unchanged"

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
Version before/after bad write: 0/0
Rows before/after bad write: 3/3
Data and commit list unchanged: True


## 4. Schema evolution (opt-in)

In [6]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
print("Schema after evolution:")
print(dt.schema())
print("\nArrow schema:")
print(dt.schema().to_arrow())
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

Schema after evolution:
Schema([Field(id, PrimitiveType("long"), nullable=True), Field(name, PrimitiveType("string"), nullable=True), Field(age, PrimitiveType("long"), nullable=True), Field(city, PrimitiveType("string"), nullable=True), Field(tier, PrimitiveType("string"), nullable=True)])

Arrow schema:
arro3.core.Schema
------------
id: Int64
name: Utf8
age: Int64
city: Utf8
tier: Utf8

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


In [7]:
commits = sorted(Path(table_path).glob("_delta_log/*.json"))
print(f"JSON commits: {len(commits)}")
for commit in commits:
    actions = [json.loads(line) for line in commit.read_text(encoding="utf-8").splitlines()]
    print(f"\n{commit.name}: actions = {[next(iter(action)) for action in actions]}")
    for action in actions:
        if "metaData" in action:
            fields = json.loads(action["metaData"]["schemaString"])["fields"]
            print("  Schema:", [(field["name"], field["type"]) for field in fields])
        if "add" in action:
            add = action["add"]
            print(f"  Added file: {add['path']} ({add['size']} bytes)")
print("\nHistory after evolution:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

JSON commits: 2

00000000000000000000.json: actions = ['commitInfo', 'protocol', 'metaData', 'add']
  Schema: [('id', 'long'), ('name', 'string'), ('age', 'long'), ('city', 'string')]
  Added file: part-00000-f4802b40-8ec7-48a7-8879-655b15bd5154-c000.snappy.parquet (1384 bytes)

00000000000000000001.json: actions = ['commitInfo', 'metaData', 'add']
  Schema: [('id', 'long'), ('name', 'string'), ('age', 'long'), ('city', 'string'), ('tier', 'string')]
  Added file: part-00000-cd410688-0691-4aa4-bb90-c9faf65d23e4-c000.snappy.parquet (1548 bytes)

History after evolution:
  v1  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 1, 'execution_time_ms': 2, 'num_retries': 0}
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 3, 'num_retries': 0}


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [8]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The schema-enforcement check uses the actual cast error and verifies that
the failed write leaves data, table version and JSON commits unchanged.

In [9]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": enforcement_blocked and bad_write_unchanged,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## Giải thích — 3.1 Delta basics

**Enforcement** kiểm tra dữ liệu ghi vào theo schema hiện tại: cột `age` là
số nguyên (`long`/`Int64`), nên chuỗi `thirty` không ép kiểu được và bị chặn.
Cell ghi sai kiểu giữ nguyên lỗi thật; assertion kiểm tra thêm rằng dữ liệu,
version và danh sách commit JSON không thay đổi. Cờ cuối notebook lấy từ các
kiểm tra này, không còn hardcoded `True`.

**Evolution** thay đổi schema có chủ đích. Lệnh append với
`schema_mode="merge"` thêm cột `tier` kiểu string, giữ `age` là số nguyên.
Ba dòng cũ có `tier = NULL`; dòng mới của dan có `tier = 'premium'`.
DuckDB vì vậy trả hai nhóm: `premium` có 1 dòng và `NULL` có 3 dòng.

**Vì sao cần opt-in?** Một cột mới có thể là lỗi gõ tên hoặc dữ liệu upstream
thay đổi ngoài ý muốn. Bật merge rõ ràng giúp kiểm soát thay đổi hợp đồng
dữ liệu với các ứng dụng đọc bảng; thêm cột không tự sửa giá trị sai kiểu.

**Transaction log chứng minh gì?** Commit v0 tạo bảng và ghi ba dòng;
commit v1 cập nhật schema và append một dòng. `metaData.schemaString` ghi
schema, `add` ghi đường dẫn Parquet, kích thước và thống kê, `commitInfo`
ghi operation, timestamp và metrics; `protocol` ở v0 ghi yêu cầu phiên bản
reader/writer. Các action trong cùng commit mô tả thay đổi đã được commit
vào snapshot. Lần ghi `thirty` thất bại không tạo commit JSON mới.
Log lưu metadata và tham chiếu file; dữ liệu hàng nằm trong Parquet.

Nguồn: [Schema validation và evolution](https://docs.delta.io/delta-batch/),
[Delta transaction log protocol](https://github.com/delta-io/delta/blob/master/PROTOCOL.md).